# EDA Note

There are business question that I'm trying to find answer

Q: Do late delivereis drive bad reviews at Olist, and how can we fix this problem ?

**Setup**

Loads the SQL extension and connects to the Olist database.

In [1]:
%load_ext sql
%sql duckdb:///../data/olist.duckdb
%config SqlMagic.displaylimit = 30
import matplotlib.pyplot as plt

Connecting to 'duckdb:///../data/olist.duckdb'

In [ ]:
%%sql

SELECT
    MIN(days_late)                 AS min_days_late,
    QUANTILE_CONT(days_late, 0.25) AS p25,
    MEDIAN(days_late)              AS median,
    QUANTILE_CONT(days_late, 0.75) AS p75,
    MAX(days_late)                 AS max_days_late
FROM orders_clean

In [31]:
%%sql

CREATE OR REPLACE VIEW orders_enriched AS
WITH main_cat AS (
    SELECT order_id, category AS main_category
    FROM (
        SELECT order_id, category, SUM(price) AS cat_revenue
        FROM items_clean
        GROUP BY order_id, category
    )
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY order_id
        ORDER BY cat_revenue DESC, category
    ) = 1
)
SELECT
    o.*,
    m.main_category,
    DATE_TRUNC('month', o.order_purchase_timestamp)::DATE AS purchase_month,
    CASE
        WHEN o.days_late <= -8 THEN '1. 8+ days early'
        WHEN o.days_late <= -1 THEN '2. 1-7 days early'
        WHEN o.days_late =   0 THEN '3. On the estimated day'
        WHEN o.days_late <=  3 THEN '4. 1-3 days late'
        WHEN o.days_late <=  7 THEN '5. 4-7 days late'
        WHEN o.days_late <= 14 THEN '6. 8-14 days late'
        ELSE                        '7. 15+ days late'
    END AS delivery_bucket,
    o.review_score <= 2 AS is_bad_review
FROM orders_clean AS o
JOIN main_cat AS m USING (order_id)


Running query in 'duckdb:///../data/olist.duckdb'

Count


In [ ]:
%%sql

SELECT
    COUNT(*)                 AS rows,
    COUNT(DISTINCT order_id) AS orders
FROM orders_enriched

Running query in 'duckdb:///../data/olist.duckdb'

rows,orders
95560,95560


In [34]:
%%sql

SELECT
    delivery_bucket,
    COUNT(*)                                AS orders,
    ROUND(100 * AVG(is_bad_review::INT), 1) AS bad_review_pct
FROM orders_enriched
GROUP BY delivery_bucket
ORDER BY delivery_bucket

Running query in 'duckdb:///../data/olist.duckdb'

delivery_bucket,orders,bad_review_pct
1. 8+ days early,70673,9.0
2. 1-7 days early,17229,10.2
3. On the estimated day,1280,12.4
4. 1-3 days late,1851,32.1
5. 4-7 days late,1748,67.7
6. 8-14 days late,1446,80.2
7. 15+ days late,1333,78.3
